# GraphKGRAGPipeline

通过「图片定位/裁剪 → OCR（含bbox）→ LLM抽取（团体/人/事件/位置）→ LPG属性图谱 → data gleaning → 社区检测/层级总结 → 查询与推理」跑通最小闭环。

> 说明：本 notebook 以“可运行 + 可扩展”为优先。
> - 没有配置 LLM/OCR 也能跑：使用 `src/extract.py` 的启发式抽取示例。
> - OCR/GraphRAG/LightRAG 属于可选集成：对应单元格可跳过。


## 1. 环境与依赖安装（GraphRAG/LightRAG + OCR/CV + 向量与图存储）

- 本项目核心依赖写在 `requirements.txt`。
- 下面演示在 notebook 里安装（可选）：你也可以在终端执行 `pip install -r requirements.txt`。


In [ ]:
# 可选：在 notebook 内安装依赖（若已在终端安装可跳过）
# %pip install -r requirements.txt

from pathlib import Path
import sys

ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print("Project root:", ROOT)


## 2. 图片定位与裁剪（版面检测/目标检测 → ROI）

这里给一个最小 ROI 示例：用 OpenCV 做简单的二值化+轮廓，输出候选框并裁剪保存。

> 如果你还没有图片素材，可以先跳过本节，直接从第 4 节用 `sample_article.md` 跑通文本→图谱。


In [ ]:
# 可选：简单 ROI 检测与裁剪（需要 opencv-python）
# 将你的图片放到 data/ 下，然后设置 IMAGE_PATH

import os
from pathlib import Path

IMAGE_PATH = Path("data") / "your_image.png"  # 改成你的图片
OUT_ROI_DIR = Path("outputs") / "rois"
OUT_ROI_DIR.mkdir(parents=True, exist_ok=True)

if IMAGE_PATH.exists():
    import cv2

    img = cv2.imread(str(IMAGE_PATH))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    thr = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 10)

    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    H, W = gray.shape[:2]
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < 800:  # 过滤小噪声
            continue
        if w < 30 or h < 15:
            continue
        boxes.append((x, y, w, h))

    boxes = sorted(boxes, key=lambda b: (b[1], b[0]))

    saved = 0
    for i, (x, y, w, h) in enumerate(boxes[:20]):
        pad = 5
        x0 = max(0, x - pad)
        y0 = max(0, y - pad)
        x1 = min(W, x + w + pad)
        y1 = min(H, y + h + pad)
        roi = img[y0:y1, x0:x1]
        out_path = OUT_ROI_DIR / f"roi_{i:02d}_{x0}_{y0}_{x1}_{y1}.png"
        cv2.imwrite(str(out_path), roi)
        saved += 1

    print("Found boxes:", len(boxes), "Saved:", saved, "->", OUT_ROI_DIR)
else:
    print("Skip ROI: image not found:", IMAGE_PATH)


## 3. OCR 与结构化文本重建（行/段落/阅读顺序）

- 本项目 `src/ocr.py` 支持两类后端：
  - `paddleocr`：返回 bbox（定位）+ 文本（更符合你的“图片识别定位”）
  - `pytesseract`：更轻，但默认拿不到 bbox

下面演示调用 `src.ocr`：输出 `ocr_text.txt` 与 `ocr_debug.json`（含 bbox）。


In [ ]:
from pathlib import Path
import json

from src.ocr import extract_ocr_spans, spans_to_text, spans_to_debug_json

IMAGE_FOR_OCR = Path("data") / "your_image.png"  # 改成你的图片
OUT = Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

if IMAGE_FOR_OCR.exists():
    spans = extract_ocr_spans(str(IMAGE_FOR_OCR))
    text = spans_to_text(spans)

    (OUT / "ocr_text.txt").write_text(text, encoding="utf-8")
    (OUT / "ocr_debug.json").write_text(
        json.dumps(spans_to_debug_json(spans), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("Wrote:", OUT / "ocr_text.txt")
    print("Wrote:", OUT / "ocr_debug.json")
    print("Preview:\n", text[:200])
else:
    print("Skip OCR: image not found:", IMAGE_FOR_OCR)


## 4. LLM 信息抽取：LPG Schema（团体/人/事件/位置）与三元组生成

- Schema 定义见 `src/schema.py`
- 抽取逻辑见 `src/extract.py`
  - 默认：启发式抽取（无API也能跑通）
  - 可选：启用 LLM，输出严格 JSON（Pydantic 校验）


In [ ]:
from src.llm import load_llm_config
from src.extract import extract_graph

text_path = Path("data") / "sample_article.md"
text = text_path.read_text(encoding="utf-8")

llm = load_llm_config(enable_llm=False)  # 改成 True 并配置 .env 可启用 LLM
extracted = extract_graph(text, llm)

print("entities:", len(extracted.entities))
print("relations:", len(extracted.relations))
print("evidence:", extracted.evidence)

# 预览前几个实体/关系
for e in extracted.entities[:5]:
    print("-", e.type, e.name, e.id)
for r in extracted.relations[:8]:
    print("-", r.type, r.source, "->", r.target)


## 5. Data Gleaning：去重、对齐、置信度、来源溯源（Provenance）

`src/gleaning.py` 先实现了一个“最小可解释”的规则推理：
- `GROUP hosted EVENT` + `EVENT held_at LOCATION` ⇒ 推断 `GROUP active_in LOCATION`
- `PERSON attended EVENT` + `EVENT held_at LOCATION` ⇒ 推断 `PERSON visited LOCATION`

后续你要更强的 gleaning：可以把“冲突消解/同名合并/置信度”做成独立模块，并把每条边的 evidence 记录到 props。


In [ ]:
from src.graph_store import GraphStore
from src.gleaning import apply_gleaning

store = GraphStore.from_extracted(extracted)
logs = apply_gleaning(store.g)

print("gleaning logs:")
for ln in logs:
    print("-", ln)

print("nodes:", store.g.number_of_nodes(), "edges:", store.g.number_of_edges())


## 6. 文档→知识图谱构建：节点/边写入（Property Graph）

这里用 NetworkX MultiDiGraph 作为本地 LPG（属性图谱），并可持久化为 `outputs/graph.json`。

> 若要换 Neo4j：把 `GraphStore.dump_json` 改成写入 Cypher/neo4j driver 即可。


In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)

store.dump_json(str(out_dir / "graph.json"))
print("Wrote:", out_dir / "graph.json")


## 7. GraphRAG 跑通：索引构建、检索、基于子图的回答生成

本项目先用“GraphRAG 核心要素”跑通：
- 图谱（LPG）
- 社区（community）
- 层级总结（hierarchy summary）
- 图查询 + 语义检索（作为召回/重排）

如果你要接入某个具体的 GraphRAG 实现（比如微软 GraphRAG），请告诉我你用的是哪一个 pip 包/仓库名，我可以把 `src/integrations/graphrag_runner.py` 继续补齐到可直接跑索引。


In [ ]:
# 可选：检查你环境里是否安装了 graphrag（仅检查，不会跑索引）
import subprocess, sys

subprocess.run([sys.executable, "-m", "src.integrations.graphrag_runner", "--check"], check=False)


## 8. LightRAG 跑通：轻量索引、混合检索（向量+图）、快速问答

同理：LightRAG 生态里不同实现的 API 差异很大，因此先做“可选接入检查”。

你如果明确是哪个 LightRAG（pip 包名/仓库地址），我就能把 runner 改成真实可运行的 indexing + query。


In [ ]:
# 可选：检查你环境里是否安装了 lightrag（仅检查，不会跑索引）
import subprocess, sys

subprocess.run([sys.executable, "-m", "src.integrations.lightrag_runner", "--check"], check=False)


## 9. 文章对应知识图谱对齐：doc_id 分区、引用回链与可视化导出

当前 demo 以单文档为例；实际落地建议：
- 为每篇文章分配 `doc_id`
- 对每个实体/关系记录 `provenance`：`doc_id/page/bbox/text_span/model_version`
- 需要可视化时：导出 GraphML/JSON，再用 pyvis/Gephi。


## 10. 查询与推理：图查询（Cypher/Gremlin）+ LLM 约束推理（补全额外信息）

- 结构化查询：用图邻域、路径、过滤条件完成精确检索
- 自然语言查询：先召回候选实体，再做 k-hop 扩展组成“证据子图”
- 约束推理：只允许基于证据子图生成结论，并输出路径/边序列作为推理链路

下面用 `QueryEngine` 演示“超越纯文本匹配”的节点语义检索 + 邻域展开。


In [ ]:
from src.query import QueryEngine

qe = QueryEngine(store)

print("semantic search (TF-IDF fallback):")
for nid, score in qe.semantic_search_nodes("读书会 上海", top_k=5):
    print("-", f"{score:.3f}", store.g.nodes[nid].get("name"), nid)

# 选一个实体看看邻域（例如 星火读书会）
ids = qe.find_by_name("星火读书会")
print("find_by_name hits:", ids)
if ids:
    nb = qe.neighborhood(ids[0], hops=2)
    print("neighborhood nodes:")
    for nid, name in nb["nodes"]:
        print("-", name, nid)
    print("neighborhood edges:")
    for u, v, t in nb["edges"]:
        print("-", t, ":", u, "->", v)


## 11. 社区检测与层级化总结：Louvain/Leiden → 社区摘要 → 多层推理

- 这里先用 NetworkX 的 greedy modularity 做社区划分（无需额外依赖）
- 产物：
  - `outputs/communities.json`
  - `outputs/hierarchy_summary.md`

后续你要 Louvain/Leiden：可以加 `python-louvain` 或 `leidenalg`，并把 `src/community.py` 的算法替换掉。


In [ ]:
from src.community import detect_communities, build_hierarchy_summary, dump_communities

communities = detect_communities(store.g)
node_to_comm = {}
for idx, nodes in enumerate(communities):
    for n in nodes:
        node_to_comm[n] = idx

summary = build_hierarchy_summary(store.g, communities, llm)

(out_dir / "hierarchy_summary.md").write_text(summary.get("_top", ""), encoding="utf-8")
dump_communities(str(out_dir / "communities.json"), communities, node_to_comm)

print("communities:", len(communities))
print("Wrote:", out_dir / "communities.json")
print("Wrote:", out_dir / "hierarchy_summary.md")
print("\nTOP SUMMARY:\n")
print(summary.get("_top", ""))


## 12. 超越文本匹配：结构相似度/路径证据/反事实检查与评测（R@k、nDCG）

最小版（本项目已具备）：
- 语义召回：`QueryEngine.semantic_search_nodes`（TF-IDF fallback，可替换成 embedding）
- 图扩展：`neighborhood(k-hop)`

你要更“GraphRAG味”的评测/反事实：
- 先准备一组问题-答案-证据集（gold）
- 对比：纯向量 vs 向量+图扩展 vs 先社区再下钻
- 反事实：移除关键边/证据后再回答，看答案稳定性
